# 03. 챗 앱 운영 구조 모의 실습

목표: Gradio 챗 앱에서 필요한 session-local history, streaming 응답, rate limit 방어, request validation을 표준 라이브러리로 구현한다.

실행 방법:
1. 이 노트북을 위에서 아래로 실행한다.
2. 실제 Gradio나 NVIDIA API를 호출하지 않는다.
3. 출력되는 mock stream과 validation error를 보며 운영 설계를 익힌다.

이 노트북은 DebuggerCafe 앱의 운영상 중요한 부분을 축소한 것이다.

In [ ]:
import time
from collections import deque
from dataclasses import dataclass, field


SUPPORTED_MIME = {"image/png", "image/jpeg", "video/mp4", "audio/mpeg", "audio/wav"}
MAX_FILES = 3
MAX_TEXT_CHARS = 4000
MAX_FILE_BYTES = 25 * 1024 * 1024

## 1. 입력 검증

입문용 데모에서는 파일을 그대로 API로 보내기 쉽다. 하지만 실제 앱에서는 파일 개수, 크기, MIME type, 텍스트 길이를 먼저 제한해야 한다.

In [ ]:
@dataclass
class UploadedFile:
    name: str
    mime: str
    size_bytes: int


def validate_user_input(text, files):
    files = files or []
    if not text.strip() and not files:
        raise ValueError("text or media input is required")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError("text is too long for this demo")
    if len(files) > MAX_FILES:
        raise ValueError("too many files")

    for file in files:
        if file.mime not in SUPPORTED_MIME:
            raise ValueError(f"unsupported MIME type: {file.mime}")
        if file.size_bytes > MAX_FILE_BYTES:
            raise ValueError(f"file is too large: {file.name}")
    return True


validate_user_input("Describe the uploaded image", [UploadedFile("chart.png", "image/png", 120_000)])

## 2. session-local history

DebuggerCafe 글의 앱은 현재 Gradio session 안에서만 history를 유지한다. 아래 클래스도 프로세스 메모리 안에만 대화를 저장한다.

In [ ]:
@dataclass
class ChatSession:
    max_turns: int = 6
    messages: deque = field(default_factory=deque)

    def add_user(self, content):
        self.messages.append({"role": "user", "content": content})
        self._trim()

    def add_assistant(self, content):
        self.messages.append({"role": "assistant", "content": content})
        self._trim()

    def _trim(self):
        # turn 수가 늘면 오래된 맥락이 API 비용을 키운다. 데모에서는 최근 메시지만 유지한다.
        while len(self.messages) > self.max_turns * 2:
            self.messages.popleft()

    def as_list(self):
        return list(self.messages)


session = ChatSession(max_turns=2)
session.add_user("What is in this video?")
session.add_assistant("It appears to show a product demo.")
session.as_list()

## 3. streaming 응답 모의 구현

원문 앱은 streaming 응답을 받아 UI에 점진적으로 표시한다. 실제 SDK 대신 mock client로 같은 제어 흐름을 연습한다.

In [ ]:
class MockNvidiaClient:
    def stream_chat(self, messages, enable_thinking=False, use_audio_in_video=False):
        last_user = next((msg for msg in reversed(messages) if msg["role"] == "user"), None)
        prefix = "Reasoned answer" if enable_thinking else "Short answer"
        audio_note = " with video audio enabled" if use_audio_in_video else ""
        response = f"{prefix}{audio_note}: I received {len(messages)} message(s). Latest user input was summarized."
        for word in response.split():
            yield word + " "


def chat_once(session, client, text, files=None, enable_thinking=False, use_audio_in_video=False):
    validate_user_input(text, files or [])
    user_content = {"text": text, "files": [file.name for file in files or []]}
    session.add_user(user_content)

    partial = ""
    for chunk in client.stream_chat(session.as_list(), enable_thinking, use_audio_in_video):
        partial += chunk
        yield partial

    session.add_assistant(partial.strip())


client = MockNvidiaClient()
for partial in chat_once(session, client, "Summarize this recording", [UploadedFile("meeting.mp4", "video/mp4", 2_000_000)], True, True):
    pass
print(partial)

## 4. 간단한 rate limit 방어

원문은 무료 NVIDIA API에 rate limit이 있다고 설명한다. 아래는 최근 요청 시간을 저장해 분당 요청 수를 제한하는 작은 limiter다.

In [ ]:
class RateLimiter:
    def __init__(self, max_calls, window_seconds):
        self.max_calls = max_calls
        self.window_seconds = window_seconds
        self.calls = deque()

    def allow(self, now=None):
        now = now if now is not None else time.time()
        while self.calls and now - self.calls[0] > self.window_seconds:
            self.calls.popleft()
        if len(self.calls) >= self.max_calls:
            return False
        self.calls.append(now)
        return True


limiter = RateLimiter(max_calls=3, window_seconds=60)
print([limiter.allow(now=0), limiter.allow(now=1), limiter.allow(now=2), limiter.allow(now=3)])
print("after window:", limiter.allow(now=70))

## 5. reasoning mode 운영 정책

reasoning mode는 복잡한 분석에는 유리할 수 있지만, 비용과 latency가 커진다. 데모 앱에서는 사용자가 직접 켜도록 하고, 파일이 큰 경우에는 기본적으로 끄는 정책을 둘 수 있다.

In [ ]:
def choose_reasoning_mode(user_requested, files):
    total_size = sum(file.size_bytes for file in files or [])
    if total_size > 10 * 1024 * 1024:
        return False, "disabled for large media to control latency"
    if user_requested:
        return True, "enabled by user"
    return False, "default concise mode"


print(choose_reasoning_mode(True, [UploadedFile("small.wav", "audio/wav", 1_000_000)]))
print(choose_reasoning_mode(True, [UploadedFile("long.mp4", "video/mp4", 20_000_000)]))

## 6. 실패 케이스 확인

검증 로직은 정상 경로보다 실패 경로에서 더 중요하다. 아래는 unsupported file과 빈 입력을 차단하는 예시다.

In [ ]:
cases = [
    ("", []),
    ("Read this file", [UploadedFile("document.pdf", "application/pdf", 500_000)]),
    ("Analyze", [UploadedFile("huge.mp4", "video/mp4", 100_000_000)]),
]

for text, files in cases:
    try:
        validate_user_input(text, files)
        print("ok")
    except ValueError as error:
        print("blocked:", error)

## 정리

- session-local history는 간단하지만 영구 저장과 개인정보 정책이 없다.
- streaming UI는 chunk를 누적해 partial answer를 갱신하는 구조다.
- production 앱은 MIME, 크기, 요청 빈도, reasoning mode 비용을 방어해야 한다.
- 실제 NVIDIA API 앱으로 확장할 때는 mock client 위치에 OpenAI-compatible client 호출을 넣으면 된다.